In [ ]:
import numpy as np                  

import matplotlib.pyplot as plt     

X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]], dtype=float)

GATES = {
    'AND': np.array([0, 0, 0, 1], dtype=float),  

    'OR':  np.array([0, 1, 1, 1], dtype=float),  

    'XOR': np.array([0, 1, 1, 0], dtype=float),  

}

def step(z):
    return (z >= 0).astype(float)      

def step_deriv(z):

    return np.zeros_like(z)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))       

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)                 

def relu(z):
    return np.maximum(0, z)            

def relu_deriv(z):
    return (z > 0).astype(float)      

ACTIVATIONS = {
    'Step':    (step,    lambda z: np.ones_like(z)),  

    'Sigmoid': (sigmoid, sigmoid_deriv),
    'ReLU':    (relu,    relu_deriv),
}

class SinglePerceptron:

    def __init__(self, act_name, lr=0.5, epochs=5000, seed=42):

        self.act_name = act_name
        self.act, self.dact = ACTIVATIONS[act_name]  

        self.lr = lr
        self.epochs = epochs
        self.seed = seed

    def fit(self, X, y):

        rng = np.random.default_rng(self.seed)

        self.W = rng.uniform(-1, 1, X.shape[1])   

        self.b = float(rng.uniform(-1, 1))         

        self.loss_history = []  

        for _ in range(self.epochs):

            z = X @ self.W + self.b    

            a = self.act(z)            

            err = y - a                

            delta = err * self.dact(z)

            self.W += self.lr * X.T @ delta / len(y)  

            self.b += self.lr * delta.mean()           

            self.loss_history.append(np.mean(err ** 2))

        return self  

    def predict(self, X):

        z = X @ self.W + self.b
        return self.act(z)

    def predict_class(self, X):

        return (self.predict(X) >= 0.5).astype(int)

    def errors(self, X, y):

        return int(np.sum(self.predict_class(X) != y.astype(int)))

    def report(self, gate_name):

        print(f"  [{self.act_name}]  W={self.W.round(4)}, b={self.b:.4f}  "
              f"→ model: {self.act_name}({self.W[0]:.3f}·x1 "
              f"+ {self.W[1]:.3f}·x2 + {self.b:.3f})")

class TwoLayerMLP:

    def __init__(self, act_name, hidden=4, lr=0.5, epochs=10000, seed=42):

        self.act_name = act_name
        self.act, self.dact = ACTIVATIONS[act_name]
        self.hidden = hidden
        self.lr = lr
        self.epochs = epochs
        self.seed = seed

    def fit(self, X, y):

        rng = np.random.default_rng(self.seed)
        n_in = X.shape[1]   

        self.W1 = rng.uniform(-1, 1, (n_in, self.hidden))
        self.b1 = np.zeros(self.hidden)

        self.W2 = rng.uniform(-1, 1, (self.hidden, 1))
        self.b2 = np.zeros(1)

        Y = y.reshape(-1, 1)   

        self.loss_history = []

        for _ in range(self.epochs):

            z1 = X  @ self.W1 + self.b1   

            a1 = self.act(z1)              

            z2 = a1 @ self.W2 + self.b2   

            a2 = sigmoid(z2)              

            err = Y - a2                  

            self.loss_history.append(np.mean(err ** 2))

            d2 = err * sigmoid_deriv(z2)              

            d1 = (d2 @ self.W2.T) * self.dact(z1)    

            self.W2 += self.lr * a1.T @ d2 / len(Y)  

            self.b2 += self.lr * d2.mean(axis=0)

            self.W1 += self.lr * X.T @ d1 / len(Y)   

            self.b1 += self.lr * d1.mean(axis=0)

        return self

    def predict(self, X):

        a1 = self.act(X @ self.W1 + self.b1)         

        a2 = sigmoid(a1 @ self.W2 + self.b2)         

        return a2.flatten()                           

    def predict_class(self, X):

        return (self.predict(X) >= 0.5).astype(int)

    def errors(self, X, y):

        return int(np.sum(self.predict_class(X) != y.astype(int)))

    def report(self, gate_name):

        print(f"  [{self.act_name}]")
        print(f"    Layer1 W1 (shape {self.W1.shape}):\n{self.W1.round(3)}")
        print(f"    Layer1 b1: {self.b1.round(3)}")
        print(f"    Layer2 W2: {self.W2.flatten().round(3)}")
        print(f"    Layer2 b2: {self.b2.round(3)}")

def plot_decision_boundary(ax, model, X, y, title):
    h = 0.02  

    x_min, x_max = -0.3, 1.3
    y_min, y_max = -0.3, 1.3
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    grid = np.c_[xx.ravel(), yy.ravel()]      

    Z = model.predict(grid).reshape(xx.shape)  

    ax.contourf(xx, yy, Z, levels=50, cmap='RdYlGn', alpha=0.6, vmin=0, vmax=1)

    ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=1.5)

    colors = ['#d62728' if yi == 0 else '#2ca02c' for yi in y]
    for i, (xi, ci) in enumerate(zip(X, colors)):
        ax.scatter(*xi, c=ci, s=120, edgecolors='white', linewidths=1.5, zorder=5)
        ax.text(xi[0] + 0.05, xi[1] + 0.05, str(int(y[i])), fontsize=8, zorder=6)

    errors = model.errors(X, y)
    ax.set_title(f"{title}\nErrors: {errors}", fontsize=9,
                 color='green' if errors == 0 else 'red')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    ax.grid(True, alpha=0.3)

def run_all():
    act_names  = ['Step', 'Sigmoid', 'ReLU']
    gate_names = list(GATES.keys())   

    print("=" * 65)
    print("PART 1 — SINGLE PERCEPTRON")
    print("=" * 65)

    single_models = {}   

    for gate_name, y in GATES.items():
        print(f"\n── Gate: {gate_name} ──")
        for act_name in act_names:
            m = SinglePerceptron(act_name).fit(X, y)     

            single_models[(gate_name, act_name)] = m     

            preds = m.predict_class(X)
            errs  = m.errors(X, y)
            m.report(gate_name)
            print(f"    Predictions: {preds}  |  True: {y.astype(int)}  |  Errors: {errs}")

    print("\n" + "=" * 65)
    print("PART 2 — TWO-LAYER MLP")
    print("=" * 65)

    mlp_models = {}

    for gate_name, y in GATES.items():
        print(f"\n── Gate: {gate_name} ──")
        for act_name in act_names:
            m = TwoLayerMLP(act_name).fit(X, y)
            mlp_models[(gate_name, act_name)] = m
            preds = m.predict_class(X)
            errs  = m.errors(X, y)
            m.report(gate_name)
            print(f"    Predictions: {preds}  |  True: {y.astype(int)}  |  Errors: {errs}")

    print("\n" + "=" * 65)
    print("PART 3 — ZERO-ERROR ANALYSIS SUMMARY TABLE")
    print("=" * 65)
    print(f"{'Model':<20} {'Gate':<6} {'Act':<10} {'Errors':<8} {'Zero?'}")
    print("-" * 55)

    for gate_name in gate_names:
        for act_name in act_names:
            e1 = single_models[(gate_name, act_name)].errors(X, GATES[gate_name])
            e2 = mlp_models[(gate_name, act_name)].errors(X, GATES[gate_name])
            print(f"{'SinglePerceptron':<20} {gate_name:<6} {act_name:<10} {e1:<8} {'✓' if e1==0 else '✗'}")
            print(f"{'TwoLayerMLP':<20} {gate_name:<6} {act_name:<10} {e2:<8} {'✓' if e2==0 else '✗'}")
        print()

    fig, axes = plt.subplots(3, 3, figsize=(13, 12))
    fig.suptitle("Single Perceptron — Decision Boundaries\n"
                 "(Rows = Activation, Columns = Gate)", fontsize=13, fontweight='bold')

    for row, act_name in enumerate(act_names):
        for col, gate_name in enumerate(gate_names):
            plot_decision_boundary(axes[row][col],
                                   single_models[(gate_name, act_name)],
                                   X, GATES[gate_name],
                                   f"{gate_name} | {act_name}")
    plt.tight_layout()
    plt.savefig('single_perceptron_boundaries.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved → single_perceptron_boundaries.png")

    fig, axes = plt.subplots(3, 3, figsize=(13, 12))
    fig.suptitle("Two-Layer MLP — Decision Boundaries\n"
                 "(Rows = Activation, Columns = Gate)", fontsize=13, fontweight='bold')

    for row, act_name in enumerate(act_names):
        for col, gate_name in enumerate(gate_names):
            plot_decision_boundary(axes[row][col],
                                   mlp_models[(gate_name, act_name)],
                                   X, GATES[gate_name],
                                   f"{gate_name} | {act_name}")
    plt.tight_layout()
    plt.savefig('mlp_boundaries.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved → mlp_boundaries.png")

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle("Training Loss Curves — XOR Gate\n"
                 "(Top row = Single Perceptron, Bottom row = MLP)", fontsize=13)

    colors = {'Step': '#e74c3c', 'Sigmoid': '#2ecc71', 'ReLU': '#3498db'}

    for col, act_name in enumerate(act_names):

        ax = axes[0][col]
        m  = single_models[('XOR', act_name)]
        ax.plot(m.loss_history, color=colors[act_name], linewidth=1.5)
        ax.set_title(f"Single Perceptron | XOR | {act_name}")
        ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss"); ax.grid(True, alpha=0.3)
        errs = m.errors(X, GATES['XOR'])
        ax.text(0.98, 0.95, f"Final errors: {errs}", transform=ax.transAxes,
                ha='right', va='top', color='red' if errs > 0 else 'green')

        ax = axes[1][col]
        m  = mlp_models[('XOR', act_name)]
        ax.plot(m.loss_history, color=colors[act_name], linewidth=1.5)
        ax.set_title(f"Two-Layer MLP | XOR | {act_name}")
        ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss"); ax.grid(True, alpha=0.3)
        errs = m.errors(X, GATES['XOR'])
        ax.text(0.98, 0.95, f"Final errors: {errs}", transform=ax.transAxes,
                ha='right', va='top', color='red' if errs > 0 else 'green')

    plt.tight_layout()
    plt.savefig('loss_curves.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved → loss_curves.png")

if __name__ == '__main__':
    run_all()


PART 1 — SINGLE PERCEPTRON

── Gate: AND ──
  [Step]  W=[0.1729 0.2528], b=-0.2828  → model: Step(0.173·x1 + 0.253·x2 + -0.283)
    Predictions: [0 0 0 1]  |  True: [0 0 0 1]  |  Errors: 0
  [Sigmoid]  W=[4.9128 4.9128], b=-7.4649  → model: Sigmoid(4.913·x1 + 4.913·x2 + -7.465)
    Predictions: [0 0 0 1]  |  True: [0 0 0 1]  |  Errors: 0
  [ReLU]  W=[1. 1.], b=-1.0000  → model: ReLU(1.000·x1 + 1.000·x2 + -1.000)
    Predictions: [0 0 0 1]  |  True: [0 0 0 1]  |  Errors: 0

── Gate: OR ──
  [Step]  W=[0.5479 0.1278], b=-0.0328  → model: Step(0.548·x1 + 0.128·x2 + -0.033)
    Predictions: [0 1 1 1]  |  True: [0 1 1 1]  |  Errors: 0
  [Sigmoid]  W=[5.628  5.6278], b=-2.5621  → model: Sigmoid(5.628·x1 + 5.628·x2 + -2.562)
    Predictions: [0 1 1 1]  |  True: [0 1 1 1]  |  Errors: 0
  [ReLU]  W=[0.5 0.5], b=0.2500  → model: ReLU(0.500·x1 + 0.500·x2 + 0.250)
    Predictions: [0 1 1 1]  |  True: [0 1 1 1]  |  Errors: 0

── Gate: XOR ──
  [Step]  W=[0.0479 0.0028], b=0.2172  → model: Step(0.04